In [1]:
import pandas as pd
import ollama
from pathlib import Path
from IPython.display import Markdown, display

In [2]:
# Config
SAFE_LIMIT = 8192
SINGLE_PASS_INPUT_BUDGET = 5200  # rough token estimate threshold
SINGLE_PRED_LIMIT = 220
CHUNK_PRED_LIMIT = 180
MULTI_PRED_LIMIT = 240
MODEL_NAME = "qwen3.5:9b"

CSV_PATH = Path("/mnt/c/Users/Ashira/Desktop/S_Projects/policy-insight-platform_personal/data/interim/training/speaker_turn_data.csv")
df = pd.read_csv(CSV_PATH)

<h2>System Prompts</h2>

In [3]:
sys_msg_single = """
You are writing a dashboard summary of a Singapore parliamentary transcript for business users.

Goal:
Produce a concise, factual policy summary that helps a reader quickly decide whether the item is relevant to them.

Rules:
- Focus on the policy signal first: the key measure, clarification, timeline, eligibility rule, statistic, constraint, or stated action.
- Preserve material facts such as numbers, dates, agencies, programme names, laws, eligibility conditions, and operational constraints.
- If the response is purely factual and directly answers the question without ambiguity, keep the summary very brief and do not add inferred gaps or additional commentary.
- If part of the original question was not answered, not tracked, or not stated, say so briefly.
- Do not add facts, conclusions, motives, sentiment, or implications not explicitly stated in the transcript.
- Use neutral language.
- Do not begin with "Mr X asked" unless necessary for clarity.
- Use attribution sparingly. Prefer "The Minister said..." only where needed.
- Combine related measures into compact lists instead of repeating similar points.
- Omit procedural details unless they change interpretation.
- Write 2 to 4 sentences in one paragraph.
- Aim for about 80 to 130 words.
- Output only the summary paragraph.
"""

sys_msg_extract = """
You are extracting factual notes from one chunk of a Singapore parliamentary transcript for later dashboard summarization.

Goal:
Capture only the details needed to build a concise policy summary.

Rules:
- Extract only what is explicitly stated in the chunk.
- Keep notes brief and information-dense.
- Preserve material numbers, dates, agencies, programme names, laws, eligibility conditions, timelines, operational constraints, and concrete measures.
- Identify whether a point is a question, answer, clarification, or procedural remark only if that helps interpretation.
- If the chunk shows that information was not provided, not tracked, or not answered, note that explicitly.
- Omit repetition, rhetorical phrasing, and minor procedural details.
- Do not infer motives, conclusions, policy implications, or sentiment.
- Output only the requested bullet structure.
"""

sys_msg_multi = """
You are writing a dashboard summary of a Singapore parliamentary debate from extracted notes.

Goal:
Produce a concise, factual policy summary for business users that highlights the most relevant signal from the debate.

Rules:
- Start with the key policy measure, clarification, timeline, eligibility rule, statistic, or stated action.
- Cover the main issue raised and the main response given, but do not waste words on parliamentary framing.
- Preserve material facts such as numbers, dates, agencies, programme names, laws, eligibility conditions, and operational constraints.
- If the response is purely factual and directly answers the question without ambiguity, keep the summary very brief and do not add inferred gaps or additional commentary.
- If the notes show that part of the question was not answered, not tracked, or not stated, say so briefly.
- Use neutral language.
- Do not add conclusions, motives, sentiment, implications, or policy analysis not present in the notes.
- Use attribution sparingly and only where it improves clarity.
- Combine related measures or responses into compact lists.
- Omit repetition, procedural details, and low-value restatement.
- Write 2 to 4 sentences in one paragraph.
- Aim for about 90 to 140 words.
- Output only the summary paragraph.
"""

<h2>Helper Functions</h2>

In [4]:
def word_count(text):
    return len(text.split()) if text else 0

In [5]:
def estimate_tokens(text):
    return len(text) // 4 if text else 0

In [6]:
def build_target_word_count(source_text, min_words=80, max_words=130, ratio=0.45):
    source_words = word_count(source_text)
    if source_words == 0:
        return min_words
    return min(max_words, max(min_words, int(source_words * ratio)))

In [7]:
def preprocess_debates(df):
    df = df.copy()

    # Ensure chronological sorting
    df["sort_date"] = pd.to_datetime(df["sittingDate"], format="%d-%m-%Y", errors="coerce")
    df = df.sort_values(["title", "sort_date", "segment_no"])

    debates = {}

    for (title, date_str), group in df.groupby(["title", "sittingDate"], sort=False):
        turns = "\n\n".join(
            f"{str(row['speaker']).upper()}: {str(row['speech_text']).strip()}"
            for _, row in group.iterrows()
            if pd.notna(row["speech_text"])
        )

        debates[title] = turns

    return debates

In [9]:
def get_text_chunks(text, chunk_size=18000, min_chunk_ratio=0.6):
    if not text or not text.strip():
        return []

    text = text.strip()
    chunks = []

    while len(text) > chunk_size:
        snip_idx = text.rfind("\n\n", 0, chunk_size)

        if snip_idx == -1:
            snip_idx = text.rfind("\n", 0, chunk_size)

        if snip_idx == -1:
            snip_idx = text.rfind(". ", 0, chunk_size)
            
        if snip_idx == -1 or snip_idx < int(chunk_size * min_chunk_ratio):
            snip_idx = chunk_size

        chunk = text[:snip_idx].strip()
        if chunk:
            chunks.append(chunk)

        text = text[snip_idx:].strip()

    if text:
        chunks.append(text)

    return chunks

In [10]:
def call_qwen(prompt, num_predict, system_msg="You are a helpful assistant.", context=8192):
    try:
        response = ollama.generate(
            model=MODEL_NAME,
            system=system_msg,
            prompt=prompt,
            stream=False,
            think=False,
            options={
                "num_ctx": context,
                "temperature": 0.0,
                "num_predict": num_predict,
                "top_p": 1.0,
                "repeat_penalty": 1.05,
                "stop": ["<|im_start|>", "<|im_end|>"]
            }
        )
        return response["response"].strip()
    except Exception as e:
        print(f"API Error: {e}")
        return ""


<h2>Summarization</h2>

In [11]:
def summarize_debate(title, transcript):
    if not transcript or not transcript.strip():
        raise ValueError("No transcript provided.")

    est_tokens = estimate_tokens(transcript)

    # Single-pass path
    if est_tokens <= SINGLE_PASS_INPUT_BUDGET:
        target_words = build_target_word_count(
            transcript,
            min_words=80,
            max_words=130,
            ratio=0.45
        )

        prompt = f"""
        DEBATE TITLE:
        {title}

        TASK:
        Write one dashboard-ready policy summary.
        Target length: about {target_words} words, clearly shorter than the source.

        TRANSCRIPT:
        {transcript}

        SUMMARY:
        """
        summary = call_qwen(
            prompt=prompt,
            num_predict=SINGLE_PRED_LIMIT,
            system_msg=sys_msg_single,
            context=SAFE_LIMIT
        )

        if not summary:
            raise ValueError("Model returned empty summary in single-pass mode.")

        return summary

    # Multi-pass path
    chunks = get_text_chunks(transcript)
    partial_notes = []

    for chunk in chunks:
        chunk_prompt = f"""
        DEBATE TITLE:
        {title}

        TASK:
        Extract only the key facts needed for a concise dashboard summary.

        TRANSCRIPT CHUNK:
        {chunk}

        Return bullets in this format:
        - Main issue or question:
        - Direct answer or position stated:
        - Key facts / figures / dates:
        - Measures, rules, programmes, or actions mentioned:
        - Missing / not stated / not tracked:
        """
        notes = call_qwen(
            prompt=chunk_prompt,
            num_predict=CHUNK_PRED_LIMIT,
            system_msg=sys_msg_extract,
            context=SAFE_LIMIT
        )

        if notes:
            partial_notes.append(notes)

    if not partial_notes:
        raise ValueError("No extraction notes generated from transcript chunks.")

    combined_text = "\n\n".join(partial_notes)

    final_target_words = build_target_word_count(
        transcript,
        min_words=90,
        max_words=140,
        ratio=0.40
    )

    final_prompt = f"""
    DEBATE TITLE:
    {title}

    TASK:
    Write one dashboard-ready policy summary based only on the extracted notes below.
    Target length: about {final_target_words} words, clearly shorter than the source.

    EXTRACTED NOTES:
    {combined_text}

    SUMMARY:
    """

    summary = call_qwen(
        prompt=final_prompt,
        num_predict=MULTI_PRED_LIMIT,
        system_msg=sys_msg_multi,
        context=SAFE_LIMIT
    )

    if not summary:
        raise ValueError("Model returned empty summary in multi-pass mode.")

    return summary


<h2>Inference</h2>

In [12]:
debates = preprocess_debates(df)

In [13]:
title = "Number of Vape-related Arrests Made, and Update on Enforcement, Preventive and Rehabilitative Measures for Tackling Vaping"
transcript = debates.get(title)
summary = summarize_debate(title, transcript)

In [14]:
Markdown(f"# Original Transcript\n## {title}\n\n{transcript}")

# Original Transcript
## Number of Vape-related Arrests Made, and Update on Enforcement, Preventive and Rehabilitative Measures for Tackling Vaping

MR ANG WEI NENG: asked the Coordinating Minister for Social Policies and Minister for Health (a) what prompted the recent inter-ministerial efforts to ban vaping; (b) how many individuals have been caught for vaping or possession offences from (i) January to June 2025 and (ii) July to August 2025; and (c) what are the Ministry's plans for the control of etomidate after its Class C classification expires on 28 February 2026.

MISS RACHEL ONG: asked the Coordinating Minister for Social Policies and Minister for Health (a) how many vaping cases have been reported via (i) hotline and (ii) webform in 2025; (b) how many of such cases led to the identification of offenders; (c) how many of these are students; and (d) whether vape use has declined since the hotline and webform were introduced.

MISS RACHEL ONG: asked the Coordinating Minister for Social Policies and Minister for Health whether the Ministry can provide an update on the enforcement actions taken against the illegal sales of Kpods to Singapore residents through various online messaging platforms.

DR WAN RIZAL: asked the Coordinating Minister for Social Policies and Minister for Health what additional preventive and rehabilitative measures will be introduced to deter youth vaping and to support those already affected.

MR MELVIN YONG YIK CHYE: asked the Coordinating Minister for Social Policies and Minister for Health (a) how many persons have been arrested and charged with vape-related offences in the past three months; and (b) what percentage of such persons are below 25 years old; and (c) what are the recent measures by the authorities to stop the sale of e-vaporisers and components on encrypted messaging platforms.

MR ONG YE KUNG: Mr Speaker, may I have your permission to answer Question Nos 7 to 11 together? My response will also address written Parliamentary Questions (PQs), Question Nos 15 to 18 in today’s Order Paper, and similar questions raised by Mr Zhulkarnain Abdul Rahim 1 , Ms Poh Li San, Mr Sharael Taha, Dr Hamid Razak and Mr David Hoe 2 , scheduled for later Sittings. I would invite Members to seek clarifications, if need be. If the question has been addressed, it may not be necessary for Members to proceed with the questions for future Sittings.

MR ONG YE KUNG: E-vaporisers, which were primarily nicotine delivery devices, were banned in Singapore ever since it was introduced. In the first eight months of this year, the Health Sciences Authority (HSA) detected almost 10,000 cases of possession or use and 38 cases of supply of e-vaporisers. Amongst users, more than half are under 25 years old. However, e-vaporisers now carry more dangerous substances, like etomidate, and controlled drugs. This was in fact one of the key considerations when we banned the supply and use of e-vaporisers years ago, and our fears have unfortunately come true. This year, HSA detected 70 cases of possession or use of etomidate e-vaporisers. With e-vaporisers, the landscape of substance abuse has changed. We launched a whole-of-Government response, which we announced on 28 August 2025. The measures include the following. First, curbing upstream import and supply. Over 2,800 online advertisements, including those found on messaging platforms, were removed between January and August this year. Platforms have a legal responsibility to detect and remove advertisements. Those with inadequate processes will face penalties, including fines and imprisonment, under the Tobacco (Control of Advertisements and Sale) Act. Second, enhance border enforcement. The HSA, Immigration and Checkpoints Authority (ICA), Central Narcotics Bureau (CNB) and Singapore Police Force (SPF) work closely to enforce against smuggling and syndicated activities at our borders. They are also sharing information with their foreign counterparts. Within the Association of Southeast Asian Nations (ASEAN) today, Singapore, Brunei, Thailand, Laos and Cambodia have banned vaping. We are pleased to learn that Malaysia is making plans to do so too. With more regional countries banning e-vaporisers, we can be more effective in curbing this harmful and addictive habit.

MR ANG WEI NENG: Thank you, Speaker, for allowing me to ask supplementary questions. In this House, I have asked many PQs about vaping since 2021. I have some supplementary questions. The first question is more on statistics. How many vaping equipment and vaping materials have been surrendered through the "bins" and through voluntary purposes since they started programme? And secondly, how many people have come voluntarily to HSA for help, because they have already surrendered their vaping equipment and need rehabilitation? And third question, is, moving on, which Ministry or department will continue the effort against vaping? I noted that the recent HSA's annual report, in the front page, report by the Chairman, there is no mention about vaping, as I mentioned the last round in my supplementary question previously.

MR ONG YE KUNG: Sir, as mentioned in my reply just now, 6,000 vapes and their components have been surrendered through the "Bin The Vape" programme. More have been surrendered earlier but we did not keep track of them. As for volunteers who gave up vapes since we announced the new policies, 74 have come forward, as I mentioned in my reply. Which Ministry will coordinate between the Ministry of Home Affairs (MHA) and MOH, we will work closely together to coordinate. The landscape has changed. It used to be either you smoke and vape, or you take drugs. Two separate sets of people. But now you have a vape device, it is a delivery device that contains the drugs. The front end is controlled by MOH, controlled substance is regulated by MHA. So, the landscape has changed. The two Ministries have to work closely together and coordinate this.

MR MELVIN YONG YIK CHYE: Thank you, Mr Speaker. I would like to ask the Minister, in his answer, he said, for vaping, there is a high number of youths caught for vaping related offences. According to news reports, these youths are enticed to vaping and peddling of vapes through social media channels. I would like to ask how well is our HSA resourced to counter such tactics employed by the vape trafficking syndicates?

MR ONG YE KUNG: I think the challenge is not one of resource, but through the Internet and messaging platforms, they proliferate very easily and they do not necessarily have a presence in Singapore for you to able to get hold of them and take them to task. So, it is a big cat and mouse game. We will continue to do so. We have shut down so many of them – I reported it in my reply – and working together with the Info-communications Media Development Authority (IMDA), as well as the tech platforms. But most importantly, the culture of avoiding substance abuse must take root in our society. What has been very comforting is that ever since we launched this new whole-of-Government approach to tackling etomidate vaping, the public has been very supportive. Schools, institutions, SSAs and even the youths amongst themselves are very supportive. So, we hope we continue to uphold this behaviour and this culture, and I think that is the best defence against substance abuse.

MS POH LI SAN: In view of the tightened measures to eradicate the use of vapes in Singapore, this has also caused a shift to more underground activities. What are the Government's measures to target this group of sales of vapes, especially Kpods, in underground activities?

MR ONG YE KUNG: Never think that just because we are worried about underground activities, so therefore, legalise and regulate. That is what other countries have done when it comes to controlled drugs; and look where they are, and look where we are. So, I think we have always taken a tough stance against drugs. I do not know why I am answering this, but I think it is a correct approach to take. Things do go underground and MHA, with our tough laws, will take very tough actions against them. So, likewise for etomidate. Do not fall for harm reduction arguments, ban it, enforce it; if it goes underground, we will go after you.

ASSOC PROF JAMUS JEROME LIM: Thank you, Speaker. Much obliged. I understand that I did not file a PQ on this, but I am just wondering if the enhanced stance against vaping will now, in fact, empower the SPF and not just the HSA, to apprehend individuals reported to be using or distributing vapes? I ask because the previous appeals I had written on behalf of residents to the SPF had been redirected. The concern however, in doing so, is two-fold. The presence of HSA officers, presumably is not as ubiquitous as that of the Police and HSA may not potentially have sufficient resources to tackle the surge in cases. This is something that Member Melvin Yong had also raised previously. Perhaps it is not within your Ministry, Sir, but I am wondering if we could have your thoughts on that.

MR ONG YE KUNG: We coordinate all these. In fact, more than the SPF. Today, SPF, HSA, ICA, CNB, the National Environment Agency, Land Transport Authority, MOE, Gambling Regulatory Authority, all the officers are empowered now to take actions against etomidate related offences. As the Member mentioned, there were instances in the past, because etomidate fell between the legal cracks, which is why there was a point that the SPF did not feel they were empowered to enforce; yet, it falls under HSA. But HSA finds that our penalties, if we catch someone possessing or using etomidate, the penalties are not stringent enough, which is why we took the step of listing etomidate as a Class C drug in MDA. So, we closed the loophole. But this is temporary. As I mentioned, early next year, we want to pass new legislation to make the necessary provision to enforce against this.

MR CHRISTOPHER DE SOUZA: I thank the Minister for his clear response. I just like to re-introduce the subutex experience all those years ago. I remember we had to fill the gap with subutex and it took some time to fill the gap for subutex before we could classify it as an illegal drug and then take enforcement action. I just see a little bit of similarity here with etomidate. I am concerned at the length of time it took HSA to classify this as a Class C drug. I do not know the ins and outs of HSA. There could be a number of reasons why the experiments had to take so long or the policy considerations. I would like to know whether we had learnt from the subutex situation and whether or not we could have been quicker in reducing, the amount of time it took to classify etomidate as a Class C. Because only then, can you come in and do all the enforcement.

MR ONG YE KUNG: The classification of etomidate under MDA is done under MHA. But I take the Member's point. We have to act as fast as we can. The process does take some time when the new substance enters a community. You have to observe who is taking it, will it fizzle out or will it start to take root and grow. Ketamine, for example, did not take root. At the same time, we have to observe the clinical evidence. How harmful is it? How addictive is it? And how dependent does someone become? So, all these take a bit of time. But I take the Member's point. We have to act as quickly as we can.

MR KENNETH TIONG BOON KIAT: There is a case in my division, where the child is hooked on Kpods and expelled from school. He refuses all contacts with authorities. His parents want to know what powers do the authorities have to enforce compulsory assessment and rehabilitation for these youths that refuse our help?

MR ONG YE KUNG: The Member wrote to me about this case. This is one of those cases that fell through the cracks during that transition period. Right now, with etomidate listed in MDA, we close the loophole. Next year, we will have new legislation.

MS HANY SOH: Thank you, Speaker. I have two supplementary questions. One is echoing from what Parliamentary colleague, Mr Christopher de Souza has said, I would like to ask whether, apart from the etomidate-laced vapes, has the Ministry detected the use of other unlawful or harmful substances as well, that may potentially be also classified?

MR ONG YE KUNG: On the first question, I assume the Member is asking whether there are other substances that are delivered through e-vaporisers. We have seen them in other countries – cannabis, for example; in Australia, we saw fentanyl. So, we have to observe this carefully and, as Mr de Souza mentioned, be very vigilant, list them quickly, do not allow them to take root. As for the next question, about MOE, I think I will answer on behalf of MOE, their measures are over and above what the law has provided. If a student is caught using etomidate through e-vaporisers, he will be required to go through the rehabilitation programme, either at IMH or one of our SSA partners. On top of that, MOE will, through their own in-house school discipline system, impose other punishments, such as suspension.

MR DENNIS TAN LIP FONG: Minister, two supplementary questions. Would it be possible to include etomidate's precursors under the precursors control list? Secondly, how do we ensure that the medical alternatives for etomidate do not get abused by drug addicts as an easy substitute?

MR ONG YE KUNG: For precursors, I assume you meant the derivatives of etomidate. They are all listed as part of the Class C drug under MDA, they have already been listed. But if we have missed out something, do let us know and we will include them. The second question, I think we just have to be tough on all forms of substance abuse so that one does not become the substitute of the other. This is an area where MHA will obviously know a lot better. But right now, as I have mentioned, e-vaporisers have changed the landscape. Now, you have got something very recreational, a delivery device – from delivering lychee-flavoured nicotine, to etomidate to cannabis. So, you have a range of stuff that can be delivered through this device. When someone is using a vape, you do not know what he or she is consuming. Because of that, MHA and MOH will have to work closely together, so that we cover all bases. So, whatever substance abuse is out there, we take firm action against them, so that one does not become a substitute for the other.

DR HAMID RAZAK: I would like to ask the Minister if Singapore has raised the issue of e-vapour product control in ASEAN forums and what have the responses from our neighbours been and whether any collaborations have emerged thus far?

MR ONG YE KUNG: We have had some informal consultations and we hope ASEAN, at some point, can take a concerted stance. This is something that we want to work towards. In the meantime, we do see more and more ASEAN countries banning vapes, Malaysia being the latest to attempt to do so, and we will share our information and our experiences with them. Hopefully, at some point, we will have a stronger ASEAN common stance.

MR VICTOR LYE: Minister, to the extent that smoking is precursor to vaping and with the clampdown on vaping, does the Ministry intend to take a tighter stance against smoking, especially among our young?

MR ONG YE KUNG: The good thing is that smoking prevalence has been dropping in Singapore continuously. The latest survey showed that prevalence among adults – "adults" meaning aged 15 and above – is 8.8%. It is a historical low and one of the lowest in the world. We will continue to encourage people not to smoke. And Singaporeans are responding to it. We should also remember that tobacco has been around for 200 years. When it first started, people did not know the harm that it can cause. Today, we know but it has taken root and become part of society's behaviour. But we will have to continue to work on reducing the prevalence of smoking.

DR WAN RIZAL: Thank you, Mr Speaker, for your indulgence. Minister, you mentioned the involvement of IMH in these efforts to reduce our vaping issues. We clearly know IMH is very much involved with rehabilitation and cessation programmes too. With this expansion, how will it affect IMH's operational ability, given that we do have many cases of mental health also requiring attention? And especially since we see a surge in youth numbers coming forward, which is a good thing, but it also affects the system. Secondly, are there plans for IMH to design dedicated youth-focused programmes apart from the adult ones, because we know that they respond differently to such therapeutic approaches?

MR ONG YE KUNG: IMH does a lot of things, with different departments. So, the department for cessation, which is called National Addictions Management Service (NAMS), have their dedicated resources. So, they will not be using resources that are used to treat or to help mental illness cases. That said, we are just starting our operations. If there is a need for more resources, we will beef them up. As for youths, we are working closely with the Ministry of Social and Family Development and the SSAs, such as WeCare and Fei Yue, who have experience working with youths. They are already starting operations on cessation programmes for youths.

In [15]:
Markdown(f"# Parliamentary Summary\n## {title}\n\n{summary}")

# Parliamentary Summary
## Number of Vape-related Arrests Made, and Update on Enforcement, Preventive and Rehabilitative Measures for Tackling Vaping

The Coordinating Minister for Social Policies and Minister for Health announced a whole-of-Government response launched on 28 August 2025 to curb vaping, citing that over half of users are under 25 and 70 etomidate cases were detected this year. Key measures include removing over 2,800 online advertisements, enhancing border enforcement with agencies like HSA and ICA, and empowering multiple ministries including MHA and MOH to act against etomidate offences. The Ministry confirmed that 6,000 vapes were surrendered via the "Bin The Vape" programme and 74 individuals sought voluntary help. While new legislation to close legal loopholes is expected early next year, current enforcement relies on existing Class C drug classifications for etomidate. Regional collaboration is ongoing, with several ASEAN nations already banning vaping.